# CODe Qwen2.5-VL-7B — ZERO-SHOT ONLY

Notebook độc lập. Không train. Có thể resume từ output của lần chạy zero-shot trước nếu attach output cũ làm Input.

Output: `/kaggle/working/stage_zero_shot_7b/`

In [1]:
from pathlib import Path
import os, sys, subprocess, json, ast, re, time, random, shutil, math
import importlib, importlib.metadata as mdlib

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SEED = 42
random.seed(SEED)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: nvidia-smi not found.")

Sun Oct 11 04:39:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
def find_full_asset():
    candidates = []
    for p in INPUT.rglob("cls_train.json"):
        root = p.parent.parent
        if (
            (root / "data" / "cls_test.csv").exists()
            and (root / "wheelhouse").exists()
        ):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy code_full_offline_assets. "
            "Hãy Add Input output/dataset từ notebook 00 FULL."
        )
    return candidates[0]

ASSET = find_full_asset()
DATA_DIR = ASSET / "data"
WHEEL_DIR = ASSET / "wheelhouse"
TRAIN_JSON = DATA_DIR / "cls_train.json"
TEST_CSV = DATA_DIR / "cls_test.csv"

print("ASSET:", ASSET)
print("TRAIN:", TRAIN_JSON)
print("TEST :", TEST_CSV)
print("WHEELS:", WHEEL_DIR)

ASSET: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets
TRAIN: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_train.json
TEST : /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_test.csv
WHEELS: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse


In [3]:
# ---------- Offline environment setup ----------
# PEFT 0.21 checks optional torchao. Kaggle currently may ship torchao 0.10,
# which is too old for PEFT 0.21 and crashes LoRA injection.
# We do not use torchao quantization, so remove it BEFORE importing peft.
try:
    torchao_version = mdlib.version("torchao")
    print("Found torchao:", torchao_version, "-> uninstalling (not needed for this pipeline)")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        check=False, text=True
    )
except mdlib.PackageNotFoundError:
    print("torchao not installed: OK")

wheels = sorted(WHEEL_DIR.glob("*.whl"))

def wheel_dist_name(p):
    return p.name.split("-", 1)[0].replace("_", "-").lower()

wheel_map = {}
for w in wheels:
    wheel_map.setdefault(wheel_dist_name(w), []).append(w)

wanted = [
    "transformers", "peft", "accelerate", "qwen-vl-utils",
    "huggingface-hub", "safetensors", "tokenizers", "av"
]
selected = []
for name in wanted:
    xs = wheel_map.get(name, [])
    if xs:
        selected.append(sorted(xs)[-1])

print("Selective offline wheels:")
for w in selected:
    print(" -", w.name)

if selected:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-index", "--no-deps", "--upgrade",
         *[str(w) for w in selected]],
        check=True
    )

# Verify in a fresh subprocess before expensive work.
preflight_code = r"""
import importlib.metadata as md
from PIL import Image
import torch, torchvision, transformers, peft, accelerate
from qwen_vl_utils import process_vision_info
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel
print("PREFLIGHT_OK")
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
try:
    print("torchao", md.version("torchao"))
except md.PackageNotFoundError:
    print("torchao", "NOT_INSTALLED")
"""
pf = subprocess.run([sys.executable, "-c", preflight_code],
                    text=True, capture_output=True)
print(pf.stdout)
if pf.returncode != 0:
    print(pf.stderr)
    raise RuntimeError("Environment preflight failed; dừng trước khi load model.")

import numpy as np
import pandas as pd
import torch
import torchvision
import transformers
import peft
import accelerate
from PIL import Image
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor,
    Trainer, TrainingArguments, TrainerCallback
)
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model, PeftModel

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this notebook.")

ENV = {
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "pillow": mdlib.version("Pillow"),
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(ENV, indent=2))

Found torchao: 0.10.0 -> uninstalling (not needed for this pipeline)
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Selective offline wheels:
 - transformers-4.57.6-py3-none-any.whl
 - peft-0.21.0-py3-none-any.whl
 - accelerate-1.15.0-py3-none-any.whl
 - qwen_vl_utils-0.0.14-py3-none-any.whl
 - huggingface_hub-0.36.2-py3-none-any.whl
 - safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/transformers-4.57.6-py3-none-any.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/peft-0.21.0-py3-none-any.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/w

In [4]:
train_data = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
test_df = pd.read_csv(TEST_CSV).fillna("")

def parse_images(x):
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x]
    s = str(x).strip()
    if not s:
        return []
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)):
            return [str(v) for v in obj]
    except Exception:
        pass
    return [v.strip() for v in s.split(",") if v.strip()]

def local_path(rel):
    rel = str(rel).replace("\\", "/").lstrip("/")
    p = ASSET / rel
    if not p.exists():
        raise FileNotFoundError(f"Missing packaged image: {rel}")
    return p

IMAGE_TOKEN_LINE = re.compile(r"(?im)^\s*image-\d+\s*:\s*<image>\s*$")

def clean_image_placeholders(text):
    s = str(text)
    s = IMAGE_TOKEN_LINE.sub("", s)
    s = s.replace("<image>", "")
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict):
                if "text" in part:
                    parts.append(str(part["text"]))
        return "\n".join(x for x in parts if x)
    return "" if content is None else str(content)

def extract_messages(item):
    user, assistant = "", ""

    msgs = item.get("messages")
    if isinstance(msgs, list):
        for m in msgs:
            role = str(m.get("role", m.get("from", ""))).lower()
            txt = content_to_text(m.get("content", m.get("value", "")))
            if role in {"user","human"} and not user:
                user = txt
            elif role in {"assistant","gpt"} and not assistant:
                assistant = txt

    conv = item.get("conversations")
    if isinstance(conv, list):
        for m in conv:
            role = str(m.get("from", m.get("role", ""))).lower()
            txt = content_to_text(m.get("value", m.get("content", "")))
            if role in {"user","human"} and not user:
                user = txt
            elif role in {"assistant","gpt"} and not assistant:
                assistant = txt

    if not user:
        user = item.get("question", item.get("q", item.get("prompt", "")))
    if not assistant:
        assistant = item.get("answer", item.get("a", item.get("label", "")))

    image_field = item.get(
        "images",
        item.get("image", item.get("photos", item.get("photographs", [])))
    )
    images = [local_path(x) for x in parse_images(image_field)]
    return clean_image_placeholders(user), content_to_text(assistant).strip(), images

lower = {str(c).lower(): c for c in test_df.columns}
TEST_IMAGES_COL = next((lower[x] for x in ["images","image","photos","photographs"] if x in lower), None)
TEST_Q_COL = next((lower[x] for x in ["question","q","prompt","user"] if x in lower), None)
TEST_A_COL = next((lower[x] for x in ["answer","a","label","assistant"] if x in lower), None)
TEST_ID_COL = next((lower[x] for x in ["id","checkup_id","patient_id"] if x in lower), None)

if TEST_IMAGES_COL is None or TEST_Q_COL is None or TEST_A_COL is None:
    raise RuntimeError("Test CSV thiếu images/question/answer aliases.")

print("Train samples:", len(train_data))
print("Test samples :", len(test_df))

Train samples: 14696
Test samples : 1793


In [5]:
def find_7b_model():
    for p in INPUT.rglob("Qwen2.5-VL-7B-Instruct"):
        if p.is_dir() and (p / "config.json").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Qwen2.5-VL-7B-Instruct. "
        "Hãy Add Input output của notebook 30_prepare_7B_model_ONLINE."
    )

MODEL_DIR = find_7b_model()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(str(MODEL_DIR), local_files_only=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR), torch_dtype=dtype,
    low_cpu_mem_usage=True, local_files_only=True
).to("cuda")
model.eval()
print("Loaded 7B:", MODEL_DIR)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded 7B: /kaggle/input/datasets/messi1231/prepare-7b/qwen7b_asset/Qwen2.5-VL-7B-Instruct


In [6]:
CLASSES = [
    "Caries", "Gingivitis", "Malocclusion",
    "Pulpitis", "Tooth Loss", "Tooth Structure Loss"
]

def parse_class_set(text):
    s = str(text).lower()
    return sorted({
        cls for cls in CLASSES
        if cls.lower() in s
    })

def safe_div(a, b):
    return float(a / b) if b else 0.0

def compute_metrics(rows):
    df = pd.DataFrame(rows).copy()
    df["true_set"] = df["true"].map(parse_class_set)
    df["pred_set"] = df["pred"].map(parse_class_set)

    exact = np.mean([a == b for a, b in zip(df["true_set"], df["pred_set"])])

    yt = np.array([[int(c in s) for c in CLASSES] for s in df["true_set"]], dtype=np.int32)
    yp = np.array([[int(c in s) for c in CLASSES] for s in df["pred_set"]], dtype=np.int32)

    tp = int(((yt == 1) & (yp == 1)).sum())
    fp = int(((yt == 0) & (yp == 1)).sum())
    fn = int(((yt == 1) & (yp == 0)).sum())

    p = safe_div(tp, tp + fp)
    r = safe_div(tp, tp + fn)
    micro_f1 = safe_div(2*p*r, p+r)

    class_f1 = []
    for j in range(len(CLASSES)):
        ctp = int(((yt[:,j] == 1) & (yp[:,j] == 1)).sum())
        cfp = int(((yt[:,j] == 0) & (yp[:,j] == 1)).sum())
        cfn = int(((yt[:,j] == 1) & (yp[:,j] == 0)).sum())
        cp = safe_div(ctp, ctp + cfp)
        cr = safe_div(ctp, ctp + cfn)
        class_f1.append(safe_div(2*cp*cr, cp+cr))

    metrics = {
        "n": int(len(df)),
        "exact_set_accuracy": float(exact),
        "micro_precision": p,
        "micro_recall": r,
        "micro_f1": micro_f1,
        "macro_f1": float(np.mean(class_f1)),
        "parsed_prediction_rate": float(np.mean([len(x) > 0 for x in df["pred_set"]])),
    }

    one = df[(df["true_set"].map(len)==1) & (df["pred_set"].map(len)==1)]
    if len(one):
        metrics["single_label_parsed_n"] = int(len(one))
        metrics["single_label_accuracy"] = float(np.mean([
            a[0] == b[0] for a,b in zip(one["true_set"], one["pred_set"])
        ]))
    return metrics, df

def build_eval_messages(row):
    refs = parse_images(row[TEST_IMAGES_COL])
    paths = [local_path(x) for x in refs]
    q = clean_image_placeholders(row[TEST_Q_COL])
    content = [{"type":"image","image":str(p)} for p in paths]
    content.append({"type":"text","text":q})
    return [{"role":"user","content":content}]

@torch.inference_mode()
def generate_batch(rows, model, processor, max_new_tokens=4096):
    msgs = [build_eval_messages(row) for row in rows]
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in msgs
    ]
    image_inputs, video_inputs = process_vision_info(msgs)
    inputs = processor(
        text=texts, images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to("cuda")

    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    trimmed = [o[len(i):] for i,o in zip(inputs.input_ids, out)]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

def find_prior_predictions(filename):
    candidates = list(INPUT.rglob(filename))
    return candidates[0] if candidates else None

def run_resumable_eval(stage_dir, filename, model, processor, batch_size=2):
    stage_dir.mkdir(parents=True, exist_ok=True)
    out_csv = stage_dir / filename

    prior = find_prior_predictions(filename)
    rows_out = []
    done = set()

    if prior:
        old = pd.read_csv(prior).fillna("")
        rows_out = old.to_dict("records")
        done = set(int(x) for x in old["row_index"].tolist())
        print("Resuming from prior predictions:", prior, "done:", len(done))

    total = len(test_df)
    pending = [i for i in range(total) if i not in done]

    for k in range(0, len(pending), batch_size):
        idxs = pending[k:k+batch_size]
        batch_rows = [test_df.iloc[i] for i in idxs]
        t0 = time.time()
        try:
            preds = generate_batch(batch_rows, model, processor)
            errs = [""] * len(idxs)
        except Exception as batch_err:
            print("Batch fallback:", repr(batch_err))
            preds, errs = [], []
            for row in batch_rows:
                try:
                    preds.append(generate_batch([row], model, processor)[0])
                    errs.append("")
                except Exception as e:
                    preds.append("")
                    errs.append(repr(e))
                torch.cuda.empty_cache()

        elapsed = time.time() - t0
        for row_index, row, pred, err in zip(idxs, batch_rows, preds, errs):
            rows_out.append({
                "row_index": row_index,
                "id": row[TEST_ID_COL] if TEST_ID_COL else row_index + 1,
                "true": row[TEST_A_COL],
                "pred": pred,
                "error": err,
                "n_images": len(parse_images(row[TEST_IMAGES_COL])),
                "seconds_batch": elapsed,
            })

        rows_out = sorted(rows_out, key=lambda x: int(x["row_index"]))
        pd.DataFrame(rows_out).to_csv(out_csv, index=False, encoding="utf-8-sig")
        completed = len(rows_out)
        if completed % 20 == 0 or completed == total:
            print(f"{completed}/{total} saved -> {out_csv.name}")
        torch.cuda.empty_cache()

    metrics, parsed = compute_metrics(rows_out)
    (stage_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    parsed.to_csv(stage_dir / "parsed_predictions.csv", index=False, encoding="utf-8-sig")
    print(json.dumps(metrics, indent=2))
    return metrics

In [7]:
STAGE = WORK / "stage_zero_shot_7b"
STAGE.mkdir(parents=True, exist_ok=True)
(STAGE / "environment.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")
metrics = run_resumable_eval(
    STAGE, "zero_shot_7b_predictions.csv", model, processor, batch_size=2
)
(STAGE / "stage_summary.json").write_text(
    json.dumps({"stage":"stage_zero_shot_7b","metrics":metrics}, indent=2),
    encoding="utf-8"
)
print("✅ stage_zero_shot_7b complete")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A

20/1793 saved -> zero_shot_7b_predictions.csv


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

40/1793 saved -> zero_shot_7b_predictions.csv


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

...

1760/1793 saved -> zero_shot_7b_predictions.csv


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

1780/1793 saved -> zero_shot_7b_predictions.csv


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


1793/1793 saved -> zero_shot_7b_predictions.csv
{
  "n": 1793,
  "exact_set_accuracy": 0.5515895147796989,
  "micro_precision": 0.4166666666666667,
  "micro_recall": 0.1171875,
  "micro_f1": 0.18292682926829265,
  "macro_f1": 0.15936407908154684,
  "parsed_prediction_rate": 0.13329615170105968,
  "single_label_parsed_n": 225,
  "single_label_accuracy": 0.4177777777777778
}
✅ stage_zero_shot_7b complete
